In [ ]:
from util.DataGen import *
from util.Plotting import *
from util.Processing import *
import numpy as np
import matplotlib.pyplot as plt

def trace_by_addition(N, kernel, energies, indeces):
    # indeces need to be sorted beforehand...
    ts = np.zeros(N)
    n = kernel.size
    assert energies.size == indeces.size
    for i, index in enumerate(indeces):
        i0 = index
        if i0 + n <= ts.size:
            i1 = i0 + n
            ts[i0:i1] += kernel * energies[i]
        else:
            # m = 0
            print('end pulse')
            ts[i0:ts.size+1] += kernel[kernel.size-(ts.size-index):] * energies[i]
    return ts

In [ ]:
# Build a trace via adding method to avoid convolution and testing the inverse...
N = 500
_, pulse = nai_pulse(1, 150)
# pulse = .25  * np.arange(15)
kernel = pulse
n = kernel.size

n_pulses = 100
# np.random.seed(0)
indeces = np.random.randint(0, N-n-1, n_pulses)
indeces = np.sort(indeces)
print(indeces)
energies = np.random.uniform(1, 3, n_pulses)
energies = 10 ** energies

energies_ts = np.zeros(N)
for i, index in enumerate(indeces):
    energies_ts[index] += energies[i]

trace_add = trace_by_addition(N, kernel, energies, indeces)
trace_fft = fft_convolve(energies_ts, kernel)
trace_td = td_convolve(energies_ts, kernel)

fig, axes = plt.subplots(3,1, figsize=(10, 8), dpi=400)
fig.tight_layout()
plot_photons(axes[0], indeces, energies, label_='Energies', color='r', alpha=.25)
axes[0].set_xlim([0, N])
axes[0].legend()
axes[0].yaxis.grid(True)

axes[1].plot(trace_td+400, label='Trace by T Domain')
axes[1].plot(trace_fft+200, label='Trace by F Domain')
axes[1].plot(trace_add, label='Trace by Addition')
axes[1].legend()
axes[1].yaxis.grid(True)
axes[1].set_title('Demonstration of Equivalence of Adding Pulses, TD and FD Convolution (Vertically Separated for Visualization)')

add_fft_error = np.abs(trace_add - trace_fft)
ft_td_error =  np.abs(trace_fft - trace_td)
td_add_error = np.abs(trace_td - trace_add)
axes[2].plot(add_fft_error, label='Add-FFT Trace Diff',alpha=.25)
axes[2].plot(ft_td_error, label='FFT-TD Trace Diff',alpha=.25)
axes[2].plot(td_add_error, label='TD-Add Trace Diff',alpha=.25)
axes[2].legend()

# Add Noise
noise_stdev = 0
trace_add_noise = trace_add + np.random.normal(0, noise_stdev, N)
trace_fft_noise = trace_fft + np.random.normal(0, noise_stdev, N)
trace_td_noise = trace_td + np.random.normal(0, noise_stdev, N)


fig, axes = plt.subplots(4,1, figsize=(10, 8), dpi=400)
fig.tight_layout()
# All Deconv via ifft
t = np.arange(N)
deconv_add = fft_deconvolve(trace_add_noise, kernel)
deconv_fft = fft_deconvolve(trace_fft_noise, kernel)
deconv_td = fft_deconvolve(trace_td_noise, kernel)

a = .1
axes[0].plot(energies_ts, label='Energies', alpha=a)
axes[0].plot(deconv_add, label='Trace by T Domain, noise_stdev={}: F Deconv'.format(noise_stdev), alpha=a)
axes[0].legend()
axes[1].plot(energies_ts, label='Energies', alpha=a)
axes[1].plot(deconv_fft, label='Trace by F Domain, noise_stdev={}: F Deconv'.format(noise_stdev), alpha=a)
axes[1].legend()
axes[2].plot(energies_ts, label='Energies', alpha=a)
axes[2].plot(deconv_td, label='Trace by Addition, noise_stdev={}: F Deconv'.format(noise_stdev), alpha=a)
axes[2].legend()
axes[2].set_xlim([0, 100])

add_error = np.abs((deconv_add - energies_ts)/energies_ts) / 100
add_error[np.isinf(add_error)] = 0
ft_error =  np.abs((deconv_fft - energies_ts)/energies_ts) / 100
ft_error[np.isinf(ft_error)] = 0
td_error = np.abs((deconv_td - energies_ts)/energies_ts) / 100
td_error[np.isinf(td_error)] = 0
axes[3].plot(add_error, label='Add Conv, FD Deconv % Diff',alpha=.25)
axes[3].plot(ft_error, label='FD Conv, FD Deconv % Diff',alpha=.25)
axes[3].plot(td_error, label='TD Conv, TD Deconv % Diff',alpha=.25)
axes[3].legend()





In [ ]:
# Study long trace!
# Looking for diff between my algo and Jeffs. Found the error it was not length, but rather diffrence in convention between
# rounding vs floor for lognormal dist decimals -> int index

N = 20000
_, pulse = nai_pulse(1, 500)
# pulse = .25  * np.arange(15)
kernel = pulse
n = kernel.size

n_pulses = 3000
# np.random.seed(0)
indeces = np.random.randint(0, N-n-1, n_pulses)
# indeces = np.random.randint(0, N, n_pulses)
indeces = np.sort(indeces)
energies = np.random.uniform(1, 3, n_pulses)
energies = 10 ** energies

energies_ts = np.zeros(N)
for i, index in enumerate(indeces):
    energies_ts[index] += energies[i]

trace_add = trace_by_addition(N, kernel, energies, indeces)
trace_fft = fft_convolve(energies_ts, kernel)


fig, axes = plt.subplots(5,1, figsize=(10, 8), dpi=400)
fig.tight_layout()
plot_photons(axes[0], indeces, energies, label_='Energies', color='r', alpha=.25)
axes[0].set_xlim([0, N])
axes[0].legend()

axes[1].plot(trace_fft+200, label='Trace by F Domain')
axes[1].plot(trace_add, label='Trace by Addition')
axes[1].legend()

add_fft_error = np.abs(trace_add - trace_fft)
axes[2].plot(add_fft_error, label='Add-FFT Trace Diff')
axes[2].legend()

# Add Noise
noise_stdev = 0
trace_fft_noise = trace_fft + np.random.normal(0, noise_stdev, N)
deconv_fft = fft_deconvolve(trace_fft_noise, kernel)
axes[3].plot(energies_ts, label='Energies', alpha=a)
axes[3].plot(deconv_fft, label='noise_stdev={}: F Deconv'.format(noise_stdev), alpha=a)
axes[3].legend()

ft_error =  np.abs((deconv_fft - energies_ts))
ft_error[np.isinf(ft_error)] = 0
axes[4].plot(ft_error, label='FD Deconv Error',alpha=.25)
axes[4].legend()

